In [1]:
import pandas as pd
import numpy as np
import warnings

In [2]:
def fill_nearest_within_range(df, group_cols, target_col, window=10):
    df = df.copy()
    
    for group_values, group_df in df.groupby(group_cols):
        valid_idx = group_df[group_df[target_col].notna()].index.to_list()
        all_idx = group_df.index.to_list()

        index_pos_map = {idx: pos for pos, idx in enumerate(all_idx)}

        for idx in all_idx:
            if pd.isna(df.at[idx, target_col]):
                pos = index_pos_map[idx]
                start_pos = max(pos - window, 0)
                end_pos = min(pos + window, len(all_idx) - 1)
                candidates = [i for i in valid_idx if start_pos <= index_pos_map[i] <= end_pos]
                if not candidates:
                    continue

                candidate_dist = [(abs(index_pos_map[c] - pos), c) for c in candidates]
                min_dist, closest_idx = min(candidate_dist, key=lambda x: x[0])
                df.at[idx, target_col] = df.at[closest_idx, target_col]

                
    return df


In [3]:
warnings.filterwarnings("ignore")

# --------------------
# 데이터 로드 및 전처리
# --------------------
data = pd.read_csv('test.csv', dtype={"evse_name": str, "station_location": str})

# UNIX timestamp → datetime (초 단위)
data["last_charge_end_time_ts"] = pd.to_datetime(data["last_charge_end_time_ts"], unit='s', errors='coerce').sort_values()

data = data.dropna(subset=["last_charge_end_time_ts", "evse_name", "station_location"])
data = data[data["last_charge_end_time_ts"] >= pd.Timestamp("2000-01-01")]




# 조합별 모든 time_idx 채우기 (Missing timestep 처리)
group_cols = ["station_location", "evse_name"]
unique_groups = data[group_cols].drop_duplicates()
df_store = pd.DataFrame()
# 그룹의 시간 범위 (min, max)
start_time = data["last_charge_end_time_ts"].min().floor('30T')
end_time = data["last_charge_end_time_ts"].max().ceil('30T')
end_time = end_time + pd.DateOffset(weeks=1)
#첫날 부터 30분씩 타임 슬롯 만들기
for idx, row in unique_groups.iterrows():
    loc = row["station_location"]
    evse = row["evse_name"]
    # 그룹별 유효 시간만 포함하여 DataFrame 생성

    # 해당 그룹의 데이터 필터링
    group_data = data[(data["station_location"] == loc) & (data["evse_name"] == evse)]  
    time_index = pd.DataFrame()
    # 30분 간격 시간 생성
    time_index['store_timestamp'] = pd.date_range(start=start_time, end=end_time, freq='30T')
    time_index['station_location'] = group_data['station_location'].iloc[0]
    time_index['evse_name'] = group_data['evse_name'].iloc[0]
     # 기존 df_store에 time_index를 이어 붙임
    df_store = pd.concat([df_store, time_index], ignore_index=True)

    print(time_index)


          store_timestamp station_location evse_name
0     2024-07-28 11:30:00         CSCS2015         0
1     2024-07-28 12:00:00         CSCS2015         0
2     2024-07-28 12:30:00         CSCS2015         0
3     2024-07-28 13:00:00         CSCS2015         0
4     2024-07-28 13:30:00         CSCS2015         0
...                   ...              ...       ...
18472 2025-08-17 07:30:00         CSCS2015         0
18473 2025-08-17 08:00:00         CSCS2015         0
18474 2025-08-17 08:30:00         CSCS2015         0
18475 2025-08-17 09:00:00         CSCS2015         0
18476 2025-08-17 09:30:00         CSCS2015         0

[18477 rows x 3 columns]
          store_timestamp station_location evse_name
0     2024-07-28 11:30:00         CV000666         0
1     2024-07-28 12:00:00         CV000666         0
2     2024-07-28 12:30:00         CV000666         0
3     2024-07-28 13:00:00         CV000666         0
4     2024-07-28 13:30:00         CV000666         0
...                 

In [4]:
df_store = df_store.sort_values('store_timestamp')
df_charge = data.sort_values('last_charge_end_time_ts')


In [5]:

# merge_asof 수행
merged_df = pd.merge_asof(
    df_store,
    df_charge,
    left_on='store_timestamp',
    right_on='last_charge_end_time_ts',
    by=['station_location', 'evse_name'],
    direction='backward',
    tolerance=pd.Timedelta('30min'),
)

# 또는 필요에 따라 특정 컬럼 삭제
# merged_df.drop(columns=[...], inplace=True)


merged_df['last_charge_end_time_ts'] = merged_df['last_charge_end_time_ts'].fillna(method='ffill')

print(merged_df)


           store_timestamp station_location evse_name last_charge_end_time_ts  \
0      2024-07-28 11:30:00         CSCS2015         0                     NaT   
1      2024-07-28 11:30:00         CV003367         1                     NaT   
2      2024-07-28 11:30:00         CV003600         0                     NaT   
3      2024-07-28 11:30:00         CV001718         0                     NaT   
4      2024-07-28 11:30:00         CV000666         1                     NaT   
...                    ...              ...       ...                     ...   
572782 2025-08-17 09:30:00         CV003600         0     2025-08-10 09:13:26   
572783 2025-08-17 09:30:00         CV001074         0     2025-08-10 09:13:26   
572784 2025-08-17 09:30:00         CV001664         0     2025-08-10 09:13:26   
572785 2025-08-17 09:30:00         CV001665         1     2025-08-10 09:13:26   
572786 2025-08-17 09:30:00         CV003608         3     2025-08-10 09:13:26   

        connection_start_ti

In [ ]:
merged_df.to_csv('data_full.csv',index=False)